# GeoFlow workflow export

Generated from a GeoFlow workflow — cells follow the node execution order; run top-to-bottom to reproduce the pipeline.

In [ ]:
import pandas as pd
import os
import geopandas as gpd
import folium
import plotly.express as px

## 1. Read CSV

type: `read_csv` — Node 1

In [ ]:
params = {
  "file_path": "/Users/lingbo/Documents/GitHub/GeoFlow/notebookflow/backend/tmp/uploads/case1_comparison_table.csv",
  "delimiter": ",",
  "encoding": "utf-8"
}
df_in = None
df_ins = [df_in]
df_out = None
html_out = None

df_out = pd.read_csv(
    params["file_path"],
    sep=params.get("delimiter", ","),
    encoding=params.get("encoding", "utf-8")
)

df_read_csv_1 = df_out
df_read_csv_1.head() if df_read_csv_1 is not None else None

## 2. GeoFile Reader

type: `geofile_reader` — Node 4

In [ ]:
params = {
  "file_path": "../examples/tl_2020_us_state-110M-continent.geojson",
  "layer": ""
}
df_in = None
df_ins = [df_in]
df_out = None
html_out = None

file_path = params.get("file_path")

if not file_path:
    raise ValueError("Please provide file_path.")

layer = params.get("layer")
# Allow GDAL to rebuild missing .shx when possible.
os.environ["SHAPE_RESTORE_SHX"] = "YES"
if layer:
    df_out = gpd.read_file(file_path, layer=layer)
else:
    df_out = gpd.read_file(file_path)

df_geofile_reader_2 = df_out
df_geofile_reader_2.head() if df_geofile_reader_2 is not None else None

## 3. Column Filter

type: `column_filter` — Node 2

In [ ]:
params = {
  "columns": [
    "Model",
    "Beta",
    "Min_est"
  ]
}
df_in = df_read_csv_1.copy()  # ← Read CSV
df_ins = [df_in]
df_out = None
html_out = None

columns = params.get("columns", [])

if not columns:
    raise ValueError("Please select at least one column.")

df_out = df_in[columns].copy()

df_column_filter_3 = df_out
df_column_filter_3.head() if df_column_filter_3 is not None else None

## 4. GeoMap

type: `geomap` — Node 5

In [ ]:
params = {
  "tiles": "OpenStreetMap",
  "zoom_start": 4
}
df_in = df_geofile_reader_2.copy()  # ← GeoFile Reader
df_ins = [df_in]
df_out = None
html_out = None

tiles = params.get("tiles", "OpenStreetMap")
zoom_start = int(params.get("zoom_start", 4))
layer_colors = ["#1976d2", "#e53935", "#2e7d32", "#f9a825", "#8e24aa",
                "#00838f", "#6d4c41", "#c2185b"]
layers = []
for i, gdf in enumerate(df_ins):
    if gdf is None:
        continue
    if "geometry" not in gdf.columns:
        raise ValueError(f"Input {i + 1} does not contain a geometry column.")
    layer = gdf[gdf.geometry.notnull()].copy()
    if layer.empty:
        continue
    layers.append((i, layer))
if not layers:
    raise ValueError("GeoMap requires at least one connected GeoDataFrame input.")
first = layers[0][1]
centroid = first.geometry.to_crs(epsg=4326).centroid
m = folium.Map(location=[float(centroid.y.mean()), float(centroid.x.mean())],
               zoom_start=zoom_start, tiles=tiles)
# Inputs are drawn bottom-to-top: port 1 is the bottom layer.
for i, layer in layers:
    color = layer_colors[i % len(layer_colors)]
    folium.GeoJson(
        layer.to_crs(epsg=4326).__geo_interface__ if layer.crs else layer.__geo_interface__,
        name=f"Layer {i + 1}",
        style_function=lambda _f, c=color: {
            "color": c, "fillColor": c, "weight": 2, "fillOpacity": 0.35,
        },
    ).add_to(m)
folium.LayerControl().add_to(m)
html_out = m.get_root().render()

if html_out:
    from IPython.display import HTML, display
    display(HTML(html_out))

## 5. Histogram

type: `histogram` — Node 3

In [ ]:
params = {
  "column": "Min_est",
  "bins": 30
}
df_in = df_column_filter_3.copy()  # ← Column Filter
df_ins = [df_in]
df_out = None
html_out = None

column = params.get("column")
bins = int(params.get("bins", 30))

if column is None:
    raise ValueError("Please select a column.")

fig = px.histogram(df_in, x=column, nbins=bins)
html_out = fig.to_html(include_plotlyjs="cdn")

if html_out:
    from IPython.display import HTML, display
    display(HTML(html_out))